# QASM file batch splitter\
\
Provide a folder and a `num_subfolders` value to split `.qasm` files evenly into a new `<folder>_split` directory.

In [2]:
from pathlib import Path
from typing import Iterable


def split_qasm_files(source_dir: Path, num_subfolders: int, suffix: str = ".qasm") -> Path:
    if num_subfolders <= 0:
        raise ValueError("num_subfolders must be > 0")

    source_dir = Path(source_dir).expanduser().resolve()
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Source directory not found: {source_dir}")

    files = sorted(p for p in source_dir.iterdir() if p.is_file() and p.name.endswith(suffix))
    if not files:
        raise FileNotFoundError(f"No {suffix} files found in {source_dir}")

    dest_dir = source_dir.with_name(f"{source_dir.name}_split")
    dest_dir.mkdir(exist_ok=True)

    subdirs = []
    for i in range(num_subfolders):
        subdir = dest_dir / f"batch_{i:03d}"
        subdir.mkdir(parents=True, exist_ok=True)
        subdirs.append(subdir)

    for idx, path in enumerate(files):
        target_dir = subdirs[idx % num_subfolders]
        target_path = target_dir / path.name
        target_path.write_bytes(path.read_bytes())

    return dest_dir


def preview_distribution(files: Iterable[Path], num_subfolders: int) -> None:
    for idx, path in enumerate(files):
        print(f"{path.name} -> batch_{idx % num_subfolders:03d}")

In [ ]:
# Example usage
source = Path("../inputs/qasm_files/quantinuum-compiled-circuits/qasm_be_tosplit")
num_subfolders = 10

dest = split_qasm_files(source, num_subfolders)
print(f"Split files into: {dest}")

Split files into: /home/linus/code/ionshuttler/inputs/qasm_files/quantinuum-compiled-circuits/qasm_be_tosplit_split


In [1]:
# Copy files whose stem contains a substring
from pathlib import Path
import shutil

def copy_files_by_stem_substring(source_dir: Path, dest_dir: Path, substring: str) -> int:
    source_dir = Path(source_dir).expanduser().resolve()
    dest_dir = Path(dest_dir).expanduser().resolve()
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Source directory not found: {source_dir}")
    dest_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for path in source_dir.iterdir():
        if not path.is_file():
            continue
        if substring in path.stem:
            shutil.copy2(path, dest_dir / path.name)
            count += 1
    return count

# Example usage
keyword = "qubits-6"
source = Path("../inputs/qasm_files/quantinuum-compiled-circuits/qasm_be_tosplit")
dest = Path(f"../inputs/qasm_files/quantinuum-compiled-circuits/qasm_be_{keyword}")
copied = copy_files_by_stem_substring(source, dest, keyword)
print(f"Copied {copied} files")


Copied 4150 files
